# 20 · DMID corrected reference assignment (v2) + re-scoring of all 190 saved models (no training)
**Runtime: T4 GPU** (about 3–4 h; resumable). Read-only for all run folders and for the v1 crops. Writes `data/external/dmid_mass_256_jit_v2/roi/` and `results/p2_dmid_v2/`.

Why: the visual audit of the v1 overlays (18c/18d, blind to model output) showed that the v1 proximity rule can give one listed mass the components of a *neighbouring* listed mass, because a component was assigned to every listed mass within 1.5 r. **Corrected rule (v2), fixed before any re-scoring:** a connected component is assigned only to the listed mass(es) with the smallest normalised distance d/r (d = Euclidean distance from the listed centre to the component's bounding box, 0 if inside), provided d/r ≤ 1.5; exact ties (e.g. one component containing two listed centres) are assigned to all tied masses. Everything else is unchanged (radius fallback 50 px, minimum 20 px, 8-connectivity, union of assigned components, duplicates = identical full-resolution component set within an image, first record in annotation order kept, same randomised crop per ROI identifier).

Steps: (1) v2 assignment + comparison with v1; (2) v2 crops (unchanged records must reproduce the v1 crop exactly — checked); (3) overlay sheets of the records whose reference changed or that are new; (4) evaluation of every saved best checkpoint on the v2 DMID cohort with the same metrics and code-replacement tests as notebook 14; (5) slim export; (6) auto-shutdown.

In [ ]:
!pip -q install imagecodecs
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, json, time, re, gc, hashlib, subprocess, requests
import numpy as np, pandas as pd, cv2, tifffile
from PIL import Image
PROJECT  = '/content/drive/MyDrive/quantum-placement'
CBIS_DIR = f'{PROJECT}/data/processed/cbis_mass_256_jit'
V1_DIR   = f'{PROJECT}/data/external/dmid_mass_256_jit/roi'
V2_DIR   = f'{PROJECT}/data/external/dmid_mass_256_jit_v2/roi'
V1_ASSIGN = f'{PROJECT}/results/p2_dmid_audit18b/assignments.csv'
OUT      = f'{PROJECT}/results/p2_dmid_v2'
SETS     = {'train': f'{PROJECT}/results/p2_jit_runs_train', 'val': f'{PROJECT}/results/p2_jit_runs_val'}
for d in (f'{V2_DIR}/images', f'{V2_DIR}/masks', f'{OUT}/sheets', f'{OUT}/eval/train', f'{OUT}/eval/val'): os.makedirs(d, exist_ok=True)
CODE_DIR = f'{PROJECT}/code'; QSEG_DIR = '/content/drive/MyDrive/quantum-placement/code'
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True); assert hits, 'qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
import importlib, tensorflow as tf, qseg2; importlib.reload(qseg2)
from qseg.config import Config
cfg = Config(work_dir=OUT)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 1 · DMID download and parsing (identical to notebooks 11 and 18c)
RAW, LOCAL = '/content/dmid_raw', '/content/dmid'; os.makedirs(RAW, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
for f in art['files']:
    if not any(k in f['name'] for k in ('TIFF', 'ROI', 'Metadata')): continue
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst) or os.path.getsize(dst) != f['size']:
        with requests.get(f['download_url'], stream=True, timeout=900) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip') and not os.path.isdir(f"{LOCAL}/{f['name'][:-4]}"):
        subprocess.run(['unzip', '-q', '-o', dst, '-d', LOCAL], check=False)
def read_tif(p):
    try: a = tifffile.imread(p)
    except Exception: a = np.array(Image.open(p))
    a = np.asarray(a)
    if a.ndim == 3 and a.shape[0] in (1, 3, 4) and a.shape[-1] not in (1, 3, 4): a = np.moveaxis(a, 0, -1)
    if a.ndim == 3: a = a[..., :3].mean(-1) if a.shape[-1] >= 3 else a[..., 0]
    return a
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
CROP_SIZE, SCALE_LO, SCALE_HI, SHIFT = 256, 1.5, 3.0, 0.8
def jit_crop(img, mask, rid, size=CROP_SIZE):
    ys, xs = np.where(mask > 0)
    if ys.size == 0: return None, None, None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    L = max(y1 - y0, x1 - x0)
    rng = np.random.default_rng(int(hashlib.md5(rid.encode()).hexdigest()[:8], 16))
    scale = rng.uniform(SCALE_LO, SCALE_HI); side = max(int(L * scale), 64)
    slack_y, slack_x = (side - (y1 - y0)) / 2, (side - (x1 - x0)) / 2
    dy, dx = rng.uniform(-SHIFT, SHIFT, 2)
    cy = int(round((y0 + y1) / 2 + dy * slack_y)); cx = int(round((x0 + x1) / 2 + dx * slack_x))
    H, W = mask.shape; top, left = cy - side // 2, cx - side // 2
    pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad:
        img = np.pad(img, pad, mode='constant'); mask = np.pad(mask, pad, mode='constant'); top += pad; left += pad
    ic = cv2.resize(img[top:top + side, left:left + side], (size, size), interpolation=cv2.INTER_AREA)
    mc = cv2.resize(mask[top:top + side, left:left + side].astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    mc = (mc > 0).astype(np.uint8) * 255
    return ic, mc, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side, scale=round(float(scale), 4), shift_y=round(float(dy), 4),
                        shift_x=round(float(dx), 4), padded=bool(pad), mask_frac=round(float((mc > 0).mean()), 4))
info_path = glob.glob(f'{LOCAL}/**/Info.txt', recursive=True)[0]
rows = []
for line in open(info_path, errors='ignore'):
    p = line.split()
    if len(p) >= 3 and p[0].upper().startswith('IMG'):
        def _f(s):
            try: return float(s)
            except ValueError: return np.nan
        rows.append(dict(img=p[0].upper(), tissue=p[1], cls=p[2], sev=p[3] if len(p) > 3 else None,
                         x=_f(p[4]) if len(p) > 6 else np.nan, y=_f(p[5]) if len(p) > 6 else np.nan, r_raw=_f(p[6]) if len(p) > 6 else np.nan))
info = pd.DataFrame(rows)
IMG_DIR = glob.glob(f'{LOCAL}/**/TIFF Images', recursive=True)[0]; ROI_DIR = glob.glob(f'{LOCAL}/**/ROI Masks', recursive=True)[0]
mass = info[info.cls.isin(['CIRC', 'SPIC', 'MISC']) & info.x.notna() & info.y.notna()].copy()
mass['r'] = mass.r_raw.fillna(50).clip(lower=20); mass = mass.reset_index(drop=True)
mass['roi_id'] = [f'{a}_{i}' for i, a in zip(mass.index, mass.img)]
print('listed masses with coordinates:', len(mass), '| images:', mass.img.nunique())

In [ ]:
# 2 · v2 (competitive) assignment, dedup, crops; v1 comparison and reproduction check
V1 = pd.read_csv(V1_ASSIGN); V1['comp_ids'] = V1.comp_ids.fillna('').astype(str)
v1ids = dict(zip(V1.roi_id, V1.comp_ids)); v1stat = dict(zip(V1.roi_id, V1.status))
def bbox_dist(st, x, y):
    x0, y0, w, h = st[:4]; dx = max(x0 - x, 0, x - (x0 + w - 1)); dy = max(y0 - y, 0, y - (y0 + h - 1)); return float(np.hypot(dx, dy))
REC = []
for img_id, grp in mass.groupby('img', sort=True):
    f = f'{ROI_DIR}/{img_id}.tif'
    if not os.path.exists(f):
        for _, r in grp.iterrows(): REC.append(dict(roi_id=r.roi_id, img=img_id, cls=r.cls, sev=r.sev, tissue=r.tissue, comps_v2='', status_v2='skipped: no mask file'))
        continue
    m = read_tif(f) > 0; im = read_tif(f'{IMG_DIR}/{img_id}.tif').astype(np.float32)
    if im.shape != m.shape: m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    D = {(r.roi_id, k): bbox_dist(st[k], r.x, r.y) / r.r for _, r in grp.iterrows() for k in range(1, n)}
    owner = {}
    for k in range(1, n):
        cand = {rid: D[(rid, k)] for rid in grp.roi_id if D[(rid, k)] <= 1.5}
        if cand:
            best = min(cand.values()); owner[k] = [rid for rid, v in cand.items() if abs(v - best) < 1e-9]
    im8 = None; seen = {}
    for _, r in grp.iterrows():
        comps = sorted(k for k, o in owner.items() if r.roi_id in o)
        rec = dict(roi_id=r.roi_id, img=img_id, cls=r.cls, sev=r.sev, tissue=r.tissue, x=r.x, y=r.y, r=r.r, comps_v2=' '.join(map(str, comps)), n_comp_v2=len(comps))
        if not comps: rec['status_v2'] = 'skipped: no component assigned'
        elif tuple(comps) in seen: rec['status_v2'] = 'duplicate (removed)'; rec['dup_of'] = seen[tuple(comps)]
        else:
            seen[tuple(comps)] = r.roi_id; rec['status_v2'] = 'retained'
            if im8 is None: im8 = to_uint8(im)
            ic, mc, mi = jit_crop(im8, np.isin(lab, comps), r.roi_id)
            cv2.imwrite(f'{V2_DIR}/images/{r.roi_id}.png', ic); cv2.imwrite(f'{V2_DIR}/masks/{r.roi_id}.png', mc); rec.update(mi)
        REC.append(rec)
    del m, im, lab, im8; gc.collect()
A2 = pd.DataFrame(REC)
A2['comps_v1'] = A2.roi_id.map(v1ids).fillna(''); A2['status_v1'] = A2.roi_id.map(v1stat)
A2['changed'] = (A2.comps_v1 != A2.comps_v2) | (A2.status_v1 != A2.status_v2)
# reproduction check: retained in both and same components -> crop must equal the v1 crop byte for byte
same = A2[(A2.status_v2 == 'retained') & (A2.status_v1 == 'retained') & ~A2.changed]
def md5(p): return hashlib.md5(open(p, 'rb').read()).hexdigest()
A2['repro_identical'] = np.nan
for i, r in same.iterrows():
    a = cv2.imread(f'{V1_DIR}/masks/{r.roi_id}.png', 0); b = cv2.imread(f'{V2_DIR}/masks/{r.roi_id}.png', 0)
    ai = cv2.imread(f'{V1_DIR}/images/{r.roi_id}.png', 0); bi = cv2.imread(f'{V2_DIR}/images/{r.roi_id}.png', 0)
    A2.loc[i, 'repro_identical'] = bool(np.array_equal(a, b) and np.array_equal(ai, bi))
A2.to_csv(f'{OUT}/assignments_v2.csv', index=False)
A2[A2.status_v2 == 'retained'].drop(columns=['comps_v1', 'status_v1']).rename(columns={'n_comp_v2': 'n_comp'}).assign(patient=lambda d: d.img).to_csv(f'{V2_DIR}/metadata.csv', index=False)
print(pd.crosstab(A2.status_v1.fillna('-'), A2.status_v2).to_string())
print('retained v2:', (A2.status_v2 == 'retained').sum(), '| changed records:', int(A2.changed.sum()),
      '| unchanged retained reproduced exactly:', int(A2.repro_identical.sum()), 'of', len(same))

In [ ]:
# 3 · Overlay sheets of changed or new records (v1 components cyan dashed-like thin, v2 components red); blind to model output
import matplotlib.pyplot as plt
CH = A2[A2.changed & ((A2.status_v1 == 'retained') | (A2.status_v2 == 'retained'))].copy()
def panel(ax, r):
    m = read_tif(f'{ROI_DIR}/{r.img}.tif') > 0; im = read_tif(f'{IMG_DIR}/{r.img}.tif')
    if im.shape != m.shape: m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8); im8 = to_uint8(im); H, W = im8.shape
    c1 = [int(k) for k in str(r.comps_v1).split()]; c2 = [int(k) for k in str(r.comps_v2).split()]
    xs = [r.x - 1.6 * r.r, r.x + 1.6 * r.r]; ys = [r.y - 1.6 * r.r, r.y + 1.6 * r.r]
    for k in set(c1) | set(c2): xs += [st[k][0], st[k][0] + st[k][2]]; ys += [st[k][1], st[k][1] + st[k][3]]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2; half = max(160, 0.55 * max(max(xs) - min(xs), max(ys) - min(ys)))
    x0, x1 = int(np.clip(cx - half, 0, W)), int(np.clip(cx + half, 0, W)); y0, y1 = int(np.clip(cy - half, 0, H)), int(np.clip(cy + half, 0, H))
    rgb = cv2.cvtColor(np.ascontiguousarray(im8[y0:y1, x0:x1]), cv2.COLOR_GRAY2RGB); s = 360 / max(rgb.shape[:2]); rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    sub = cv2.resize(lab[y0:y1, x0:x1].astype(np.float32), (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST).astype(np.int32)
    for k in np.unique(sub):
        if k == 0: continue
        cs, _ = cv2.findContours((sub == k).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        col = (255, 40, 40) if k in c2 else ((0, 200, 255) if k in c1 else (150, 150, 150))
        cv2.drawContours(rgb, cs, -1, col, 2 if k in c2 else 1)
        yy, xx = np.nonzero(sub == k); cv2.putText(rgb, str(k), (int(xx.mean()), int(yy.mean())), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
    c = (int((r.x - x0) * s), int((r.y - y0) * s)); cv2.circle(rgb, c, max(1, int(r.r * s)), (255, 230, 0), 1); cv2.drawMarker(rgb, c, (255, 230, 0), cv2.MARKER_CROSS, 10, 1)
    ax.imshow(rgb); ax.axis('off')
    ax.set_title(f"{r.roi_id} {r.cls}/{r.sev}\nv1 [{r.comps_v1 or '-'}] {r.status_v1} -> v2 [{r.comps_v2 or '-'}] {r.status_v2}", fontsize=5.2)
    del m, im, lab, im8, sub; gc.collect()
for j in range(0, len(CH), 24):
    part = CH.iloc[j:j + 24]; fig, axs = plt.subplots(4, 6, figsize=(18, 13))
    for a in axs.ravel(): a.axis('off')
    for a, (_, r) in zip(axs.ravel(), part.iterrows()):
        try: panel(a, r)
        except Exception as e: a.set_title(f'{r.roi_id}: drawing failed ({type(e).__name__})', fontsize=5.2)
    plt.tight_layout(); fig.savefig(f'{OUT}/sheets/changed_{j // 24 + 1:02d}.png', dpi=80); plt.close(fig); gc.collect()
print('changed records drawn:', len(CH), '| sheets:', sorted(os.listdir(f'{OUT}/sheets')))

In [ ]:
# 4 · Re-score every saved best checkpoint on the v2 DMID cohort (same procedure as notebook 14, DMID only; resumable)
meta = pd.read_csv(f'{V2_DIR}/metadata.csv')
def _png(p, mask):
    a = tf.image.decode_png(tf.io.read_file(p), channels=1).numpy()
    return (a > 127).astype(np.float32) if mask else a.astype(np.float32) / 255.0
XD = np.stack([_png(f'{V2_DIR}/images/{r}.png', False) for r in meta.roi_id]); YD = np.stack([_png(f'{V2_DIR}/masks/{r}.png', True) for r in meta.roi_id])
XC, YC, DFC = qseg2.load_cbis(CBIS_DIR)
K_DRAWS = 10
def eval_run(set_name, run_dir):
    run = os.path.basename(run_dir); dst = f'{OUT}/eval/{set_name}/{run}.csv'
    if os.path.exists(dst): return None
    s = json.load(open(f'{run_dir}/summary.json')); name, fold, seed = s['model'], int(s['fold']), int(s['seed'])
    tf.keras.backend.clear_session()
    m = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m.load_weights(f'{run_dir}/best.weights.h5')
    tr, va, te = qseg2.fold_split(DFC, fold)
    has_sever = 'sever' in [l.name for l in m.layers]
    P0 = m.predict(XD, batch_size=cfg.batch_size, verbose=0)
    t = qseg2.per_image_scores(YD, P0, cfg.threshold); t.insert(0, 'roi_id', meta.roi_id.values)
    if has_sever:
        qseg2.set_sever_bank(m, XC[tr], cfg.batch_size, seed); sev = m.get_layer('sever'); bank0 = sev.bank.numpy().copy()
        Pc = qseg2.predict_with_mode(m, XD, 1, cfg.batch_size)
        t['dice_const'] = qseg2.per_image_scores(YD, Pc, cfg.threshold).dice.values
        t['flip_const'] = ((Pc[..., 0] >= cfg.threshold) != (P0[..., 0] >= cfg.threshold)).mean((1, 2)); t['absdp_const'] = np.abs(Pc - P0)[..., 0].mean((1, 2))
        for d in range(K_DRAWS):
            perm = np.arange(len(bank0)) if d == 0 else np.random.default_rng(1000 + d).permutation(len(bank0))
            sev.bank.assign(bank0[perm]); Ps = qseg2.predict_with_mode(m, XD, 2, cfg.batch_size)
            t[f'dice_swap{d}'] = qseg2.per_image_scores(YD, Ps, cfg.threshold).dice.values
            if d == 0:
                t['flip_swap'] = ((Ps[..., 0] >= cfg.threshold) != (P0[..., 0] >= cfg.threshold)).mean((1, 2)); t['absdp_swap'] = np.abs(Ps - P0)[..., 0].mean((1, 2))
        sev.bank.assign(bank0)
    t.insert(0, 'dataset', 'DMID'); t['keep'] = True
    t['model'], t['fold'], t['seed'], t['set'] = name, fold, seed, set_name
    t['epochs_run'], t['best_epoch'] = s.get('epochs_run'), s.get('best_epoch')
    t.to_csv(dst, index=False); return t
n_new = 0
for set_name, root in SETS.items():
    runs = [d for d in sorted(glob.glob(f'{root}/*_fold*_seed*')) if os.path.exists(f'{d}/summary.json') and os.path.exists(f'{d}/best.weights.h5')]
    print(set_name, len(runs), 'runs')
    for d in runs:
        t0 = time.time(); r = eval_run(set_name, d)
        if r is not None:
            n_new += 1; print(f"{set_name:6s} {os.path.basename(d):22s} DMID v2 {r.dice.mean():.4f} ({time.time() - t0:.0f} s)", flush=True)
print('evaluated', n_new, 'new runs')

In [ ]:
# 5 · Slim export (same columns as eval_slim.csv.gz) + summary
allv = pd.concat([pd.read_csv(f) for s in SETS for f in sorted(glob.glob(f'{OUT}/eval/{s}/*.csv'))], ignore_index=True)
cols = ['set', 'model', 'fold', 'seed', 'dataset', 'roi_id', 'keep', 'dice', 'iou', 'hd95', 'dice_const', 'flip_const', 'absdp_const', 'flip_swap', 'absdp_swap'] + [f'dice_swap{d}' for d in range(10)] + ['epochs_run', 'best_epoch']
for c in cols:
    if c not in allv: allv[c] = np.nan
allv[cols].round(4).to_csv(f'{OUT}/eval_dmid_v2_slim.csv.gz', index=False, compression='gzip')
print('runs:', allv.groupby(['set', 'model', 'fold', 'seed']).ngroups, '| rows:', len(allv), '| file MB:', round(os.path.getsize(f'{OUT}/eval_dmid_v2_slim.csv.gz') / 1e6, 2))
print(allv.groupby(['set', 'model']).dice.mean().round(4).unstack(0).to_string())

In [ ]:
# 6 · Auto-shutdown: save Drive, then release the GPU
from google.colab import drive, runtime
drive.flush_and_unmount(); print('Drive flushed'); time.sleep(30); runtime.unassign()